In [3]:
!pip install -q gtts transformers librosa soundfile
!pip install datasets tqdm
!pip install ipywidgets
!pip install piper-tts
!pip install onnxruntime-gpu
!pip install torchaudio
!pip install sentence-transformers datasets

In [4]:
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    HubertModel, Wav2Vec2FeatureExtractor,
    AutoTokenizer, AutoModel
)
from datasets import load_from_disk
import librosa
from tqdm import tqdm

2026-02-28 13:21:08.889427: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772284869.085174      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772284869.136586      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772284869.616865      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772284869.616919      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772284869.616922      55 computation_placer.cc:177] computation placer alr

In [5]:
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [6]:
import torch
import librosa
import soundfile as sf
from gtts import gTTS
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [7]:
import uuid
import torchaudio
from datasets import load_dataset, Dataset

In [8]:
class CFG:
    # models
    audio_model_name  = "facebook/hubert-base-ls960"
    text_model_name   = "sentence-transformers/all-MiniLM-L6-v2"

    # embedding dims
    audio_hidden_dim  = 768   # HuBERT-base output
    proj_dim          = 384   # match MiniLM output dim

    # audio
    sample_rate       = 16000
    max_audio_len     = 5     # seconds — adjust to your clip lengths

    # training
    batch_size        = 32
    num_epochs        = 10
    lr                = 3e-5
    warmup_steps      = 500
    temperature       = 0.07  # InfoNCE temperature

    # loss weights
    lambda_cosine     = 1.0
    lambda_infonce    = 1.0

    # misc
    seed              = 42
    num_workers       = 2
    device            = "cuda" if torch.cuda.is_available() else "cpu"
    save_path         = "audio_encoder_distilled.pt"

    # set this to your dataset path on Kaggle
    # dataset_path      = "/kaggle/input/ms-marco-audio-100k-no-prefix/msmarco_queries_tts"

In [9]:
class AudioEncoderWithProjection(nn.Module):
    """
    HuBERT-base + mean pooling + projection head to 384-dim.
    Only the projection head + last N transformer layers are trained
    to stay within Kaggle memory limits.
    """
    def __init__(self, cfg, unfreeze_last_n_layers=4):
        super().__init__()
        self.hubert = HubertModel.from_pretrained(cfg.audio_model_name)

        # freeze everything first
        for param in self.hubert.parameters():
            param.requires_grad = False

        # unfreeze the last N encoder layers
        encoder_layers = self.hubert.encoder.layers
        for layer in encoder_layers[-unfreeze_last_n_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

        # projection head: 768 → 384 with a bottleneck
        self.projection = nn.Sequential(
            nn.Linear(cfg.audio_hidden_dim, cfg.audio_hidden_dim),
            nn.GELU(),
            nn.LayerNorm(cfg.audio_hidden_dim),
            nn.Linear(cfg.audio_hidden_dim, cfg.proj_dim),
        )

    def forward(self, input_values):
        # HuBERT outputs [B, T', 768]
        outputs = self.hubert(input_values=input_values)
        hidden  = outputs.last_hidden_state          # [B, T', 768]
        pooled  = hidden.mean(dim=1)                 # [B, 768]
        proj    = self.projection(pooled)            # [B, 384]
        return F.normalize(proj, dim=-1)             # L2 normalize

In [10]:
import os
import wave
from piper import PiperVoice

voice = PiperVoice.load("en_US-lessac-low.onnx", use_cuda=True)

def synthesize_and_save(query, qid, target_sr = 16000):
    path = os.path.join(f"{qid}.wav")
    if os.path.exists(path):
        return path
    audio = []
    with wave.open(path, "wb") as wav_file:
        voice.synthesize_wav(query, wav_file)
    return path

2026-02-28 13:21:26.976813958 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 28 Memcpy nodes are added to the graph torch_jit for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-02-28 13:21:26.997201446 [W:onnxruntime:, session_state.cc:1327 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-02-28 13:21:26.997237211 [W:onnxruntime:, session_state.cc:1329 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


In [11]:
checkpoint = torch.load("/kaggle/input/models/premsagars2023/kd-audio-encoder-with-all-cosine/pytorch/default/1/audio_encoder_distilled (3).pt")
audio_model = AudioEncoderWithProjection(CFG())
audio_model.load_state_dict(checkpoint["model_state"])
audio_model.eval().to(CFG().device)

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

In [12]:
@torch.no_grad()
def encode_audio(audio_path, model, feature_extractor, cfg):
    """Encode a single audio file into a 384-dim embedding."""
    audio, _ = librosa.load(audio_path, sr=cfg.sample_rate, mono=True)
    max_samples = cfg.sample_rate * cfg.max_audio_len
    if len(audio) > max_samples:
        audio = audio[:max_samples]
    else:
        audio = np.pad(audio, (0, max_samples - len(audio)))

    inputs = feature_extractor(audio, sampling_rate=cfg.sample_rate, return_tensors="pt")
    input_values = inputs.input_values.to(cfg.device)

    model.eval()
    emb = model(input_values)
    return emb.cpu().numpy()  # [1, 384]

In [13]:
# ------------------------------------------------------------
# 2. Model (single-stage pretrained retriever)
# ------------------------------------------------------------
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
# ------------------------------------------------------------
# 3. Stream datasets
# ------------------------------------------------------------
print("Loading streaming datasets...")
corpus_ds = load_dataset(
    "sentence-transformers/msmarco-corpus",
    "passage",
    split="train",
    streaming=True
)

queries_ds = load_dataset(
    "sentence-transformers/msmarco-corpus",
    "query",
    split="train",
    streaming=True
)

Loading streaming datasets...


README.md: 0.00B [00:00, ?B/s]

In [15]:
# ------------------------------------------------------------
# 4. Load qrels
# ------------------------------------------------------------
qrels = pd.read_csv(
    "/kaggle/input/qrels-ms-marco/qrels.dev.tsv",
    sep="\t",
    names=["qid", "iter", "pid", "rel"]
)

qrels = qrels[qrels["rel"] > 0]
relevant_map = qrels.groupby("qid")["pid"].apply(set).to_dict()

In [16]:
# ------------------------------------------------------------
# 5. Choose evaluation queries FIRST (FAST)
# ------------------------------------------------------------
MAX_QUERIES = 1000

eval_qids = []
for item in queries_ds:
    qid = int(item["qid"])
    if qid in relevant_map:
        eval_qids.append(qid)
    if len(eval_qids) >= MAX_QUERIES:
        break

eval_qids = set(eval_qids)

eval_relevant_pids = set()
for qid in eval_qids:
    eval_relevant_pids.update(relevant_map[qid])

print(f"Evaluation queries: {len(eval_qids)}")
print(f"Relevant passages required: {len(eval_relevant_pids)}")

Evaluation queries: 1000
Relevant passages required: 1060


In [17]:
# ------------------------------------------------------------
# 6. Build corpus in ONE streaming pass (FAST)
# ------------------------------------------------------------
TARGET_CORPUS_SIZE = 100_000

corpus_texts = []
corpus_id_map = {}
seen_pids = set()

print("Collecting corpus (positives + negatives)...")

for stream_idx, item in enumerate(corpus_ds, start=1):
    pid = int(item["pid"])

    if pid in eval_relevant_pids:
        corpus_texts.append("passage: " + item["text"])
        corpus_id_map[len(corpus_texts) - 1] = pid
        seen_pids.add(pid)

    elif len(corpus_texts) < TARGET_CORPUS_SIZE:
        corpus_texts.append("passage: " + item["text"])
        corpus_id_map[len(corpus_texts) - 1] = pid

    if stream_idx % 200000 == 0:
        print(
            f"[streamed={stream_idx:,}] "
            f"kept={len(corpus_texts):,} "
            f"positives_found={len(seen_pids):,}/{len(eval_relevant_pids):,}"
        )
    
    if (
        len(corpus_texts) >= TARGET_CORPUS_SIZE
        and (len(seen_pids) / len(eval_relevant_pids) > 0.99)
    ):
        break

print(f"Final corpus size: {len(corpus_texts)}")
print(f"All positives present: {eval_relevant_pids.issubset(seen_pids)}")

[streamed=200,000] kept=100,005 positives_found=11/1,060
[streamed=400,000] kept=100,011 positives_found=17/1,060
[streamed=600,000] kept=100,017 positives_found=23/1,060
[streamed=800,000] kept=100,022 positives_found=28/1,060
[streamed=1,000,000] kept=100,026 positives_found=32/1,060
[streamed=1,200,000] kept=100,033 positives_found=39/1,060
[streamed=1,400,000] kept=100,038 positives_found=44/1,060
[streamed=1,600,000] kept=100,043 positives_found=49/1,060
[streamed=1,800,000] kept=100,049 positives_found=55/1,060
[streamed=2,000,000] kept=100,050 positives_found=56/1,060
[streamed=2,200,000] kept=100,051 positives_found=57/1,060
[streamed=2,400,000] kept=100,057 positives_found=63/1,060
[streamed=2,600,000] kept=100,062 positives_found=68/1,060
[streamed=2,800,000] kept=100,064 positives_found=70/1,060
[streamed=3,000,000] kept=100,071 positives_found=77/1,060
[streamed=3,200,000] kept=100,076 positives_found=82/1,060
[streamed=3,400,000] kept=100,079 positives_found=85/1,060
[stre

In [18]:
# ------------------------------------------------------------
# 7. Encode corpus on GPU
# ------------------------------------------------------------
print(f"Encoding {len(corpus_texts)} passages...")
corpus_embeddings = model.encode(
    corpus_texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings = True
)

Encoding 101044 passages...


Batches:   0%|          | 0/395 [00:00<?, ?it/s]

In [19]:
# ------------------------------------------------------------
# 8. Evaluation (MRR@10)
# ------------------------------------------------------------
print("Evaluating MRR@10...")
mrr_score = 0.0
query_count = 0

# Re-stream queries (streaming datasets are one-pass)
queries_ds = load_dataset(
    "sentence-transformers/msmarco-corpus",
    "query",
    split="train",
    streaming=True
)

for idx, item in enumerate(queries_ds):
    if idx % 20_000 == 0: print(idx)
    qid = int(item["qid"])
    if qid not in eval_qids:
        continue

    audio_path = synthesize_and_save(item["text"], qid)
    query_emb = encode_audio(audio_path, audio_model, feature_extractor, CFG())
    query_emb = query_emb.squeeze(0)  # [1, 384] → [384]
    
    # tts = gTTS(item["text"], lang="en")
    # tts.save(TMP_WAV)

    # audio, sr = librosa.load(TMP_WAV, sr=16000)

    # audio = add_mild_noise_and_speed(audio, sr)

    # inputs = processor(
    #     audio,
    #     sampling_rate=16000,
    #     return_tensors="pt"
    # ).to(device)

    # with torch.no_grad():
    #     predicted_ids = asr_model.generate(inputs.input_features)

    # asr_text = processor.batch_decode(
    #     predicted_ids,
    #     skip_special_tokens=True
    # )[0]
    
    # query_emb = model.encode(
    #     "query: " + item["text"],
    #     convert_to_tensor=True
    # )

    hits = util.semantic_search(
        query_emb,
        corpus_embeddings,
        top_k=10
    )[0]

    for rank, hit in enumerate(hits):
        retrieved_pid = corpus_id_map[hit["corpus_id"]]
        if retrieved_pid in relevant_map[qid]:
            mrr_score += 1.0 / (rank + 1)
            break

    query_count += 1

print("\n--- RESULTS ---")
print(f"Evaluated queries: {query_count}")
print(f"Text-to-Text Upper Bound (MRR@10): {mrr_score / query_count:.4f}")


Evaluating MRR@10...
0
20000
40000
60000
80000
100000
120000
140000
160000
180000
200000
220000
240000
260000
280000
300000
320000
340000
360000
380000
400000
420000
440000
460000
480000
500000
520000
540000
560000
580000
600000
620000
640000
660000
680000
700000
720000
740000
760000
780000
800000
820000
840000
860000
880000
900000
920000
940000
960000
980000
1000000

--- RESULTS ---
Evaluated queries: 1000
Text-to-Text Upper Bound (MRR@10): 0.2353
